# STUDIO OM — Oficina Multimédia

Três programas Python separados por secções. Executa as células de código por ordem. A última abre o menu principal.

Utiliza um kernel Python local com ambiente gráfico (Tkinter). O Google Colab normalmente não suporta estas janelas.

### Instalação (se necessário)
Retira o `#` da linha abaixo e executa a célula para instalar as dependências.

In [ ]:
# %pip install pillow wordcloud

## Programa 01 — Nuvem de palavras

In [ ]:
# ========================================================================
# PROGRAMA 01 — NUVEM DE PALAVRAS
# ========================================================================

# STUDIO OM — Nuvem de palavras. Requer Pillow e wordcloud.
import re
import unicodedata
from collections import Counter
import tkinter as tk
from tkinter import ttk, colorchooser, filedialog, messagebox
from PIL import Image, ImageTk, ImageDraw
from wordcloud import WordCloud

DICIONARIO = {
    'amor': ['afeto', 'carinho'], 'alegria': ['felicidade', 'contentamento'],
    'vida': ['existência', 'jornada'], 'luz': ['claridade', 'brilho'],
    'esperança': ['confiança', 'otimismo'], 'mundo': ['universo', 'planeta'],
    'coração': ['alma', 'sentimento'], 'sonho': ['desejo', 'imaginação'],
    'paz': ['serenidade', 'harmonia'], 'caminho': ['percurso', 'trilho'],
    'bonito': ['belo', 'encantador'], 'tristeza': ['melancolia', 'saudade'],
    'sofrimento': ['dor', 'aflição'], 'aliviar': ['suavizar', 'amenizar'],
    'fazer': ['realizar', 'construir'], 'criar': ['inventar', 'conceber'],
    'ideias': ['conceitos', 'pensamentos'], 'cor': ['tom', 'tonalidade']
}
PALETAS = {
    'Violeta': ['#5A4FD6', '#8975EB', '#3E369C', '#9C6EE1', '#5F76CE', '#BC80DE'],
    'Viridis': ['#3D2678', '#345BA2', '#258E9D', '#35A985', '#8CBF68', '#4A7FA8'],
    'Verde': ['#1D6E63', '#429F83', '#274F64', '#6CB7A1', '#377C75'],
    'Terra': ['#8C4F4A', '#BE7854', '#7A5F88', '#D4A068', '#6F5951']
}
FORMATOS = {
    'Paisagem original': (1200, 630), 'A4 vertical': (1240, 1754),
    'A4 horizontal': (1754, 1240), 'A3 vertical': (1754, 2480),
    'Quadrado': (1200, 1200), 'Cartaz 4:5': (1080, 1350),
    'Story vertical': (1080, 1920), 'Horizontal 16:9': (1920, 1080),
    'Personalizado': None
}
PADRAO = re.compile(r"[^\W_]+(?:[’'-][^\W_]+)*", re.UNICODE)


def normalizar(texto):
    return ''.join(c for c in unicodedata.normalize('NFD', texto.lower())
                   if not unicodedata.combining(c))


def contar_palavras(texto, ignorar='', limite=20):
    palavras = PADRAO.findall(unicodedata.normalize('NFC', texto))
    excluidas = {normalizar(p.strip()) for p in ignorar.split(',')}
    frequencias = Counter()
    grafias = {}
    for palavra in palavras:
        chave = normalizar(palavra)
        if chave not in excluidas:
            frequencias[chave] += 1
            grafias.setdefault(chave, palavra.lower())
    ordem = sorted(frequencias, key=lambda p: (-frequencias[p], p))
    return len(palavras), len(frequencias), {
        grafias[p]: frequencias[p] for p in ordem[:limite or 100]}


def criar_imagem(frequencias, largura, altura, fundo, paleta):
    if not frequencias:
        imagem = Image.new('RGB', (largura, altura), fundo)
        desenho = ImageDraw.Draw(imagem)
        desenho.text((largura // 2, altura // 2), 'Escreve um texto para começar',
                     fill='#666699', anchor='mm')
        return imagem, 0
    cores = PALETAS[paleta]
    nuvem = WordCloud(width=largura, height=altura, background_color=fundo,
                      max_words=100, prefer_horizontal=1, margin=8,
                      random_state=42, collocations=False,
                      color_func=lambda *args, **kwargs: kwargs['random_state'].choice(cores))
    nuvem.generate_from_frequencies(frequencias)
    return nuvem.to_image(), len(nuvem.layout_)


class NuvemPalavras:
    def __init__(self, root):
        self.root = root
        root.title('STUDIO OM — Nuvem de palavras')
        root.geometry('1050x780')
        root.minsize(880, 690)
        root.columnconfigure(0, weight=1)
        root.rowconfigure(1, weight=1)
        ttk.Label(root, text='Dá forma às tuas ideias.', font=('Arial', 23, 'bold')).grid(row=0, column=0, columnspan=2, sticky='w', padx=20, pady=15)
        palco = ttk.Frame(root, padding=15)
        palco.grid(row=1, column=0, sticky='nsew')
        palco.columnconfigure(0, weight=1)
        palco.rowconfigure(0, weight=1)
        self.preview = tk.Label(palco, bg='#E9EDF7')
        self.preview.grid(row=0, column=0, sticky='nsew')
        self.stats = tk.StringVar()
        ttk.Label(palco, textvariable=self.stats, wraplength=480).grid(row=1, column=0, pady=10)
        ttk.Label(palco, text='Dashboard de sinónimos').grid(row=2, column=0, sticky='w')
        self.sugestoes = ttk.Frame(palco)
        self.sugestoes.grid(row=3, column=0, sticky='ew')
        painel = ttk.Frame(root, padding=15)
        painel.grid(row=1, column=1, sticky='ns')
        ttk.Label(painel, text='Personalização', font=('Arial', 15, 'bold')).pack(anchor='w')
        ttk.Label(painel, text='Texto de origem').pack(anchor='w', pady=(12, 2))
        self.texto = tk.Text(painel, width=37, height=7, wrap='word')
        self.texto.pack(fill='x')
        self.texto.bind('<KeyRelease>', lambda e: self.atualizar_sinonimos())
        self.ignorar = tk.StringVar()
        self.quantidade = tk.StringVar(value='20')
        self.paleta = tk.StringVar(value='Violeta')
        self.formato = tk.StringVar(value='Paisagem original')
        self.largura = tk.StringVar(value='1200')
        self.altura = tk.StringVar(value='630')
        self.fundo = '#EDEFFF'
        ttk.Label(painel, text='Palavras a ignorar (vírgulas)').pack(anchor='w', pady=(10, 2))
        ttk.Entry(painel, textvariable=self.ignorar).pack(fill='x')
        for titulo, variavel, valores in [
            ('Quantidade', self.quantidade, ['10', '20', '40', '80', 'Todas (até 100)']),
            ('Paleta', self.paleta, list(PALETAS)),
            ('Formato do cartaz', self.formato, list(FORMATOS))]:
            ttk.Label(painel, text=titulo).pack(anchor='w', pady=(9, 2))
            combo = ttk.Combobox(painel, textvariable=variavel, values=valores, state='readonly')
            combo.pack(fill='x')
            if variavel is self.formato:
                combo.bind('<<ComboboxSelected>>', self.mudar_formato)
        dimensoes = ttk.Frame(painel)
        dimensoes.pack(fill='x', pady=7)
        self.campos = []
        for titulo, var in [('Largura (px)', self.largura), ('Altura (px)', self.altura)]:
            coluna = ttk.Frame(dimensoes)
            coluna.pack(side='left', expand=True, fill='x')
            ttk.Label(coluna, text=titulo).pack(anchor='w')
            campo = ttk.Entry(coluna, textvariable=var, width=12, state='readonly')
            campo.pack(fill='x', padx=(0, 5))
            self.campos.append(campo)
        self.botao_cor = tk.Button(painel, text=self.fundo, bg=self.fundo, command=self.escolher_fundo)
        ttk.Label(painel, text='Fundo').pack(anchor='w')
        self.botao_cor.pack(fill='x')
        ttk.Button(painel, text='Gerar nuvem', command=self.gerar).pack(fill='x', pady=(14, 5))
        ttk.Button(painel, text='Guardar PNG', command=self.guardar).pack(fill='x')
        self.imagem = None
        self.foto = None
        self.preview.bind('<Configure>', self.mostrar)
        self.atualizar_sinonimos()
        self.gerar()

    def mudar_formato(self, event=None):
        tamanho = FORMATOS[self.formato.get()]
        for campo in self.campos:
            campo.configure(state='normal' if tamanho is None else 'readonly')
        if tamanho:
            self.largura.set(str(tamanho[0]))
            self.altura.set(str(tamanho[1]))
            self.gerar()

    def escolher_fundo(self):
        cor = colorchooser.askcolor(color=self.fundo, parent=self.root)[1]
        if cor:
            self.fundo = cor.upper()
            self.botao_cor.configure(text=self.fundo, bg=self.fundo)

    def atualizar_sinonimos(self):
        for filho in self.sugestoes.winfo_children():
            filho.destroy()
        palavras = {normalizar(p) for p in PADRAO.findall(self.texto.get('1.0', 'end-1c'))}
        sugestoes = [(p, s) for p, lista in DICIONARIO.items() if normalizar(p) in palavras for s in lista][:6]
        for i, (p, s) in enumerate(sugestoes):
            ttk.Button(self.sugestoes, text=f'{p} → {s}', command=lambda p=p, s=s: self.substituir(p, s)).grid(row=i//2, column=i%2, sticky='w', padx=3, pady=3)
        if not sugestoes:
            ttk.Label(self.sugestoes, text='Sem sugestões. Experimenta amor, luz ou esperança.').grid()

    def substituir(self, origem, alternativa):
        texto = self.texto.get('1.0', 'end-1c')
        for m in PADRAO.finditer(texto):
            if normalizar(m.group()) == normalizar(origem):
                texto = texto[:m.start()] + alternativa + texto[m.end():]
                self.texto.delete('1.0', 'end')
                self.texto.insert('1.0', texto)
                break
        self.atualizar_sinonimos()
        self.gerar()

    def gerar(self):
        try:
            w, h = int(self.largura.get()), int(self.altura.get())
            if not (320 <= w <= 4000 and 320 <= h <= 4000 and w*h <= 8000000):
                raise ValueError
        except ValueError:
            messagebox.showerror('Dimensões inválidas', 'Usa 320 a 4000 px por lado e até 8 milhões de píxeis.')
            return False
        limite = 100 if self.quantidade.get().startswith('Todas') else int(self.quantidade.get())
        total, distintas, freqs = contar_palavras(self.texto.get('1.0', 'end-1c')[:25000], self.ignorar.get(), limite)
        try:
            self.imagem, visiveis = criar_imagem(freqs, w, h, self.fundo, self.paleta.get())
        except (ValueError, OSError) as erro:
            messagebox.showerror('Nuvem de palavras', str(erro))
            return False
        self.stats.set(f'Palavras totais: {total}  |  Distintas: {distintas}\nVisíveis: {visiveis} de {len(freqs)}  |  {w} × {h} px')
        self.atualizar_sinonimos()
        self.mostrar()
        return True

    def mostrar(self, event=None):
        if self.imagem is None:
            return
        copia = self.imagem.copy()
        copia.thumbnail((max(1, self.preview.winfo_width()-12), max(1, self.preview.winfo_height()-12)))
        self.foto = ImageTk.PhotoImage(copia)
        self.preview.configure(image=self.foto)

    def guardar(self):
        if not self.gerar():
            return
        w, h = self.imagem.size
        caminho = filedialog.asksaveasfilename(defaultextension='.png', filetypes=[('Imagem PNG', '*.png')], initialfile=f'studio-om-nuvem-{w}x{h}.png')
        if caminho:
            try:
                self.imagem.save(caminho, 'PNG')
            except OSError as erro:
                messagebox.showerror('Erro ao guardar', str(erro))

## Programa 02 — Cores e formatos

In [ ]:
# ========================================================================
# PROGRAMA 02 — CORES E FORMATOS
# ========================================================================

# STUDIO OM — Cores e formatos. Usa apenas a biblioteca padrão do Python.
import colorsys
import math
import re
import tkinter as tk
from tkinter import ttk, colorchooser, messagebox

PREDEFINICOES = {'HD': (1280, 720), 'Full HD': (1920, 1080),
                 'QHD': (2560, 1440), '4K': (3840, 2160)}


def analisar_resolucao(w, h):
    if not (isinstance(w, int) and isinstance(h, int) and 1 <= w <= 100000 and 1 <= h <= 100000):
        raise ValueError('Introduz valores inteiros entre 1 e 100 000 píxeis.')
    maior, menor = max(w, h), min(w, h)
    categoria = 'Abaixo de HD'
    for nome, x, y in [('8K UHD', 7680, 4320), ('4K UHD', 3840, 2160),
                       ('QHD', 2560, 1440), ('Full HD', 1920, 1080), ('HD', 1280, 720)]:
        if maior >= x and menor >= y:
            categoria = nome
            break
    divisor = math.gcd(w, h)
    orientacao = 'Quadrado' if w == h else 'Horizontal' if w > h else 'Vertical'
    return categoria, f'{w//divisor}:{h//divisor}', w*h/1e6, orientacao


def validar_hex(cor):
    cor = '#' + cor.strip().lstrip('#').upper()
    if not re.fullmatch(r'#[0-9A-F]{6}', cor):
        raise ValueError('Usa um código HEX válido, por exemplo #547EAE.')
    return cor


def hex_rgb(cor):
    cor = validar_hex(cor)
    return tuple(int(cor[i:i+2], 16) for i in (1, 3, 5))


def hex_hsl(cor):
    h, l, s = colorsys.rgb_to_hls(*(n/255 for n in hex_rgb(cor)))
    return h*360, s*100, l*100


def limitar(n, minimo, maximo):
    return max(minimo, min(maximo, n))


def arredondar(n):
    return math.floor(n + 0.5)


def hsl_hex(h, s, l):
    rgb = colorsys.hls_to_rgb((h % 360)/360, limitar(l, 0, 100)/100, limitar(s, 0, 100)/100)
    return '#' + ''.join(f'{arredondar(n*255):02X}' for n in rgb)


def gerar_paleta(base):
    base = validar_hex(base)
    h, s, l = hex_hsl(base)
    return [hsl_hex(h-18, s*.75, limitar(l+30, 14, 92)),
            hsl_hex(h-7, s*.85, limitar(l+15, 12, 85)), base,
            hsl_hex(h+8, s*1.05, limitar(l-14, 9, 80)),
            hsl_hex(h+26, s*.80, limitar(l-25, 8, 70))]


class CoresFormatos:
    def __init__(self, root):
        self.root = root
        root.title('STUDIO OM — Cores e formatos')
        root.geometry('1000x720')
        root.minsize(850, 640)
        root.columnconfigure(0, weight=1)
        root.rowconfigure(1, weight=1)
        ttk.Label(root, text='Explora cada píxel.', font=('Arial', 24, 'bold')).grid(row=0, column=0, columnspan=2, sticky='w', padx=20, pady=15)
        palco = ttk.Frame(root, padding=15)
        palco.grid(row=1, column=0, sticky='nsew')
        palco.columnconfigure(0, weight=1)
        palco.rowconfigure(1, weight=1)
        ttk.Label(palco, text='Visualização da proporção').grid(row=0, column=0, sticky='w')
        self.canvas = tk.Canvas(palco, bg='#E9EDF7', height=280, highlightthickness=0)
        self.canvas.grid(row=1, column=0, sticky='nsew', pady=8)
        self.metricas = tk.StringVar()
        ttk.Label(palco, textvariable=self.metricas, font=('Arial', 11, 'bold')).grid(row=2, column=0, pady=6)
        ttk.Label(palco, text='Paleta de cores — seleciona um tom').grid(row=3, column=0, sticky='w', pady=(12, 5))
        self.paleta = ttk.Frame(palco)
        self.paleta.grid(row=4, column=0, sticky='ew')
        self.selecionada = '#547EAE'
        self.base = self.selecionada
        self.amostra = tk.Label(palco, text='Aa', font=('Arial', 29, 'bold'), height=2)
        self.amostra.grid(row=5, column=0, sticky='ew', pady=10)
        self.codigos = ttk.Frame(palco)
        self.codigos.grid(row=6, column=0, sticky='ew')
        self.botoes_codigo = []
        for i in range(3):
            botao = ttk.Button(self.codigos)
            botao.pack(side='left', expand=True, fill='x', padx=2)
            self.botoes_codigo.append(botao)
        painel = ttk.Frame(root, padding=15)
        painel.grid(row=1, column=1, sticky='ns')
        ttk.Label(painel, text='Configuração', font=('Arial', 16, 'bold')).pack(anchor='w')
        ttk.Label(painel, text='Resoluções rápidas').pack(anchor='w', pady=(15, 5))
        for nome, dimensoes in PREDEFINICOES.items():
            ttk.Button(painel, text=nome, command=lambda d=dimensoes: self.predefinir(d)).pack(fill='x', pady=2)
        self.largura = tk.StringVar(value='1920')
        self.altura = tk.StringVar(value='1080')
        for titulo, var in [('Largura (px)', self.largura), ('Altura (px)', self.altura)]:
            ttk.Label(painel, text=titulo).pack(anchor='w', pady=(10, 2))
            campo = ttk.Entry(painel, textvariable=var)
            campo.pack(fill='x')
            campo.bind('<Return>', lambda e: self.calcular())
        ttk.Button(painel, text='Analisar resolução', command=self.calcular).pack(fill='x', pady=12)
        ttk.Separator(painel).pack(fill='x', pady=6)
        ttk.Label(painel, text='Identificador de cores', font=('Arial', 13, 'bold')).pack(anchor='w', pady=5)
        self.hex_var = tk.StringVar(value=self.selecionada)
        ttk.Label(painel, text='Introduzir HEX').pack(anchor='w')
        entrada = ttk.Entry(painel, textvariable=self.hex_var)
        entrada.pack(fill='x')
        entrada.bind('<Return>', lambda e: self.aplicar_hex())
        entrada.bind('<KeyRelease>', lambda e: self.hex_automatico())
        ttk.Button(painel, text='Aplicar HEX', command=self.aplicar_hex).pack(fill='x', pady=3)
        ttk.Button(painel, text='Escolher cor', command=self.escolher_cor).pack(fill='x', pady=3)
        ttk.Button(painel, text='Gerar paleta com esta cor', command=self.nova_paleta).pack(fill='x', pady=3)
        self.estado = tk.StringVar()
        ttk.Label(painel, textvariable=self.estado, wraplength=230).pack(anchor='w', pady=8)
        self.dimensoes = (1920, 1080)
        self.canvas.bind('<Configure>', lambda e: self.desenhar())
        self.calcular()
        self.nova_paleta()

    def predefinir(self, dimensoes):
        self.largura.set(str(dimensoes[0]))
        self.altura.set(str(dimensoes[1]))
        self.calcular()

    def calcular(self):
        try:
            a, b = self.largura.get().strip(), self.altura.get().strip()
            if not all(re.fullmatch(r'[0-9]+', x) for x in (a, b)):
                raise ValueError('Introduz valores inteiros entre 1 e 100 000 píxeis.')
            w, h = int(a), int(b)
            categoria, proporcao, mp, orientacao = analisar_resolucao(w, h)
        except ValueError as erro:
            self.estado.set(str(erro))
            return
        self.dimensoes = w, h
        mp_texto = f'{mp:.2f}'.rstrip('0').rstrip('.').replace('.', ',')
        self.metricas.set(f'{categoria}  |  {proporcao}  |  {mp_texto} MP  |  {orientacao}')
        self.estado.set('')
        self.desenhar()

    def desenhar(self):
        w, h = self.dimensoes
        cw, ch = self.canvas.winfo_width(), self.canvas.winfo_height()
        self.canvas.delete('all')
        escala = min(max(1, cw-36)/w, max(1, ch-36)/h, 490/w, 284/h)
        pw, ph = w*escala, h*escala
        x, y = cw/2, ch/2
        self.canvas.create_rectangle(x-pw/2, y-ph/2, x+pw/2, y+ph/2, fill='#8ABBCF', outline='#202C44', width=3 if min(pw, ph) >= 8 else 1)
        if pw >= 135 and ph >= 68:
            self.canvas.create_text(x, y, text=f'{analisar_resolucao(w,h)[0]}\n{w} × {h} px', justify='center', fill='#1B3B55', font=('Arial', 15, 'bold'))

    def selecionar(self, cor):
        self.selecionada = validar_hex(cor)
        self.hex_var.set(self.selecionada)
        rgb = hex_rgb(cor)
        h, s, l = hex_hsl(cor)
        codigos = [self.selecionada, f'rgb({rgb[0]}, {rgb[1]}, {rgb[2]})',
                   f'hsl({arredondar(h)}, {arredondar(s)}%, {arredondar(l)}%)']
        self.amostra.configure(bg=self.selecionada, fg='#1F2742' if l > 52 else '#FFFFFF')
        for botao, nome, codigo in zip(self.botoes_codigo, ['HEX', 'RGB', 'HSL'], codigos):
            botao.configure(text=f'{nome}: {codigo}\nCopiar', command=lambda c=codigo: self.copiar(c))
        for botao in self.paleta.winfo_children():
            botao.configure(relief='sunken' if botao.cget('text') == self.selecionada else 'raised')

    def hex_automatico(self):
        try:
            cor = validar_hex(self.hex_var.get())
        except ValueError:
            return
        self.selecionar(cor)

    def aplicar_hex(self):
        try:
            self.selecionar(self.hex_var.get())
            self.estado.set('Cor selecionada.')
        except ValueError as erro:
            self.estado.set(str(erro))

    def escolher_cor(self):
        cor = colorchooser.askcolor(self.selecionada, parent=self.root)[1]
        if cor:
            self.selecionar(cor)

    def nova_paleta(self):
        self.base = self.selecionada
        for filho in self.paleta.winfo_children():
            filho.destroy()
        for cor in gerar_paleta(self.base):
            fg = '#1F2742' if hex_hsl(cor)[2] > 52 else '#FFFFFF'
            tk.Button(self.paleta, text=cor, bg=cor, fg=fg, height=2, command=lambda c=cor: self.selecionar(c)).pack(side='left', expand=True, fill='x', padx=2)
        self.selecionar(self.base)
        self.estado.set(f'Cor base: {self.base}')

    def copiar(self, codigo):
        self.root.clipboard_clear()
        self.root.clipboard_append(codigo)
        self.estado.set(f'{codigo} copiado.')

## Programa 03 — Jogo do número secreto

In [ ]:
# ========================================================================
# PROGRAMA 03 — JOGO DO NÚMERO SECRETO
# ========================================================================

# STUDIO OM — Jogo do número secreto. Usa apenas a biblioteca padrão.
import json
import re
import secrets
from pathlib import Path
import tkinter as tk
from tkinter import ttk

MODOS = {'Fácil': (30, True), 'Médio': (10, True), 'Difícil': (10, False)}
CHAVES = {'Fácil': 'facil', 'Médio': 'medio', 'Difícil': 'dificil'}
FICHEIRO_RECORDES = Path.home() / '.studio_om_recordes.json'


def ler_recordes():
    try:
        dados = json.loads(FICHEIRO_RECORDES.read_text(encoding='utf-8'))
        return {k: v for k, v in dados.items() if type(v) is int and v > 0} if isinstance(dados, dict) else {}
    except (OSError, ValueError):
        return {}


class Partida:
    def __init__(self, modo='Fácil', secreto=None):
        self.modo = modo
        self.limite, self.dicas = MODOS[modo]
        self.secreto = secrets.randbelow(100) + 1 if secreto is None else secreto
        self.tentativas = []
        self.minimo, self.maximo = 1, 100
        self.terminado = False
        self.venceu = False

    def tentar(self, valor):
        if self.terminado:
            return 'Fim do jogo', 'Começa uma nova partida.', 'erro'
        valor = str(valor).strip()
        if not re.fullmatch(r'[0-9]+', valor) or len(valor) > 100:
            return 'Número inválido', 'Introduz um número inteiro entre 1 e 100.', 'erro'
        palpite = int(valor)
        if not 1 <= palpite <= 100:
            return 'Número inválido', 'Introduz um número inteiro entre 1 e 100.', 'erro'
        if palpite in self.tentativas:
            return 'Número repetido', 'Esse número já foi utilizado. Experimenta outro.', 'erro'
        self.tentativas.append(palpite)
        usadas = len(self.tentativas)
        restantes = self.limite - usadas
        if palpite == self.secreto:
            self.terminado = self.venceu = True
            self.minimo = self.maximo = palpite
            return 'Fantástico! Acertaste!', f'O número era {palpite}. Conseguiste em {usadas} tentativa(s).', 'sucesso'
        if restantes == 0:
            self.terminado = True
            return 'Acabaram as tentativas', f'O número era {self.secreto}. Inicia uma nova partida.', 'erro'
        if not self.dicas:
            return 'Ainda não acertaste', f'Tenta outro número. Restam {restantes} tentativa(s).', 'normal'
        if palpite < self.secreto:
            self.minimo = max(self.minimo, palpite + 1)
            dica = 'O número secreto é maior.'
        else:
            self.maximo = min(self.maximo, palpite - 1)
            dica = 'O número secreto é menor.'
        if usadas >= 3:
            dica += f' Está entre {self.minimo} e {self.maximo}.'
        if usadas >= 6:
            dica += ' É par.' if self.secreto % 2 == 0 else ' É ímpar.'
        return 'Boa tentativa!', f'{dica} Restam {restantes} tentativa(s).', 'normal'


class JogoNumeroSecreto:
    def __init__(self, root):
        self.root = root
        self.recordes = ler_recordes()
        root.title('STUDIO OM — Jogo do número secreto')
        root.geometry('950x570')
        root.minsize(790, 500)
        root.rowconfigure(1, weight=1)
        root.columnconfigure(0, weight=1)
        ttk.Label(root, text='Consegues descobrir?', font=('Arial', 24, 'bold')).grid(row=0, column=0, columnspan=2, sticky='w', padx=20, pady=15)
        palco = tk.Frame(root, bg='#252449', padx=25, pady=25)
        palco.grid(row=1, column=0, sticky='nsew', padx=(15, 0), pady=(0, 15))
        tk.Label(palco, text='O NÚMERO SECRETO ESTÁ ENTRE', bg='#252449', fg='#C0B4FF', font=('Arial', 11, 'bold')).pack(pady=(30, 5))
        self.intervalo = tk.Label(palco, text='1 — 100', bg='#252449', fg='#F7EFC2', font=('Arial', 54, 'bold'))
        self.intervalo.pack()
        self.legenda = tk.Label(palco, bg='#252449', fg='#D1C8F4', font=('Arial', 11), wraplength=400)
        self.legenda.pack(pady=10)
        formulario = tk.Frame(palco, bg='#252449')
        formulario.pack(pady=15)
        self.entrada = ttk.Entry(formulario, width=12, justify='center', font=('Arial', 22))
        self.entrada.pack(side='left', padx=5)
        self.entrada.bind('<Return>', lambda e: self.tentar())
        self.botao = tk.Button(formulario, text='Experimentar', command=self.tentar, bg='#F98B70', fg='#272037', font=('Arial', 12, 'bold'), padx=10, pady=8)
        self.botao.pack(side='left', padx=5)
        self.feedback = tk.Label(palco, bg='#37335D', fg='white', font=('Arial', 12), wraplength=400, justify='left', padx=16, pady=16)
        self.feedback.pack(fill='x', pady=10)
        painel = ttk.Frame(root, padding=20)
        painel.grid(row=1, column=1, sticky='ns')
        ttk.Label(painel, text='Progresso do jogo', font=('Arial', 15, 'bold')).pack(anchor='w')
        ttk.Label(painel, text='Dificuldade').pack(anchor='w', pady=(15, 3))
        self.modo = tk.StringVar(value='Fácil')
        combo = ttk.Combobox(painel, textvariable=self.modo, values=list(MODOS), state='readonly', width=27)
        combo.pack(fill='x')
        combo.bind('<<ComboboxSelected>>', lambda e: self.novo_jogo())
        self.contagem = tk.StringVar()
        ttk.Label(painel, textvariable=self.contagem).pack(anchor='w', pady=(20, 5))
        self.progresso = ttk.Progressbar(painel, mode='determinate')
        self.progresso.pack(fill='x')
        ttk.Label(painel, text='Histórico').pack(anchor='w', pady=(20, 4))
        self.historico = ttk.Label(painel, wraplength=255, justify='left')
        self.historico.pack(anchor='w')
        ttk.Label(painel, text='O teu melhor resultado').pack(anchor='w', pady=(18, 3))
        self.recorde = ttk.Label(painel, font=('Arial', 17, 'bold'), foreground='#5849BA')
        self.recorde.pack(anchor='w')
        ttk.Button(painel, text='Começar novo jogo', command=self.novo_jogo).pack(fill='x', pady=16)
        self.informacao = ttk.Label(painel, wraplength=255, justify='left')
        self.informacao.pack(anchor='w')
        self.novo_jogo()

    def mensagem(self, titulo, texto, estado='normal'):
        cores = {'normal': '#37335D', 'erro': '#693D57', 'sucesso': '#28634F'}
        self.feedback.configure(text=f'{titulo}\n\n{texto}', bg=cores[estado])

    def novo_jogo(self):
        self.partida = Partida(self.modo.get())
        p = self.partida
        self.entrada.configure(state='normal')
        self.entrada.delete(0, 'end')
        self.legenda.configure(text='O número secreto está neste intervalo' if p.dicas else 'Sem dicas: tens 10 tentativas para acertar.')
        self.informacao.configure(text='Após cada tentativa, sabes se o número é maior ou menor. A partir da 3.ª, recebes o intervalo atualizado. Na 6.ª, descobres a paridade.' if p.dicas else 'Modo difícil: sem dicas de maior/menor, intervalo ou paridade.')
        self.mensagem('Pronto para jogar?', 'Escolhe um número entre 1 e 100 para começares.')
        self.atualizar()
        self.entrada.focus_set()

    def tentar(self):
        p = self.partida
        if p.terminado:
            return
        n = len(p.tentativas)
        self.mensagem(*p.tentar(self.entrada.get()))
        if len(p.tentativas) > n:
            self.entrada.delete(0, 'end')
        if p.venceu:
            chave = CHAVES[p.modo]
            resultado = len(p.tentativas)
            if resultado < self.recordes.get(chave, float('inf')):
                self.recordes[chave] = resultado
                try:
                    FICHEIRO_RECORDES.write_text(json.dumps(self.recordes), encoding='utf-8')
                except OSError:
                    self.feedback.configure(text=self.feedback.cget('text') + '\nRecorde guardado apenas nesta sessão.')
            self.legenda.configure(text='Número secreto descoberto!')
        elif p.terminado:
            self.legenda.configure(text='Fim do jogo')
        self.atualizar()
        if not p.terminado:
            self.entrada.focus_set()

    def atualizar(self):
        p = self.partida
        self.intervalo.configure(text=f'{p.minimo} — {p.maximo}')
        self.contagem.set(f'Tentativas usadas: {len(p.tentativas)} / {p.limite}')
        self.progresso.configure(maximum=p.limite, value=len(p.tentativas))
        self.historico.configure(text='  ·  '.join(map(str, p.tentativas)) or 'Ainda não jogaste.')
        recorde = self.recordes.get(CHAVES[p.modo])
        self.recorde.configure(text=f'{recorde} tentativa(s)' if recorde else '—')
        estado = 'disabled' if p.terminado else 'normal'
        self.entrada.configure(state=estado)
        self.botao.configure(state=estado)

## Menu principal

In [ ]:
# ========================================================================
# MENU PRINCIPAL — ABRIR OS TRÊS PROGRAMAS
# ========================================================================

def menu_principal():
    root = tk.Tk()
    root.title('STUDIO OM — Dashboard Oficina Multimédia')
    root.geometry('560x400')
    root.minsize(460, 350)
    root.configure(bg='#141A36')
    janelas = {}

    def abrir(nome, programa):
        janela = janelas.get(nome)
        if janela is not None and janela.winfo_exists():
            janela.deiconify()
            janela.lift()
            janela.focus_force()
            return
        janela = tk.Toplevel(root)
        janelas[nome] = janela
        programa(janela)

    tk.Label(root, text='STUDIO OM', bg='#141A36', fg='#FFA48F',
             font=('Arial', 25, 'bold')).pack(pady=(30, 6))
    tk.Label(root, text='Cria. Explora. Experimenta.', bg='#141A36',
             fg='white', font=('Arial', 15)).pack(pady=(0, 20))
    for nome, programa, cor in [
        ('01 — Nuvem de palavras', NuvemPalavras, '#FFB09B'),
        ('02 — Cores e formatos', CoresFormatos, '#A9E1E7'),
        ('03 — Jogo do número secreto', JogoNumeroSecreto, '#D6C5FF'),
    ]:
        tk.Button(root, text=nome, bg=cor, fg='#20253D',
                  font=('Arial', 13, 'bold'), pady=10,
                  command=lambda n=nome, p=programa: abrir(n, p)).pack(
                      fill='x', padx=45, pady=5)
    ttk.Button(root, text='Sair', command=root.destroy).pack(pady=14)
    root.mainloop()

## Executar o dashboard
Fecha as janelas para terminar a execução desta célula.

In [ ]:
if __name__ == '__main__':
    menu_principal()